# Phiên GPU cuối cùng trên Google Colab (A4)

Notebook này làm toàn bộ `docs/FINAL_GPU_RUNBOOK_VI.md` trên Colab: A4 (kiểm tra tái lập, 40 lần huấn luyện,
mở tập test Sleep-EDF một lần), suy luận trên Telemetry và SHHS1, rồi ghi `final_results.tar.gz` vào Drive.

**Chuẩn bị**
1. Ba tệp `a4_inputs_sleepedf.tar`, `final_core.tar`, `final_holdout.tar` nằm trong thư mục Drive
   `MyDrive/SleepTCN_upload/` (sửa `DRIVE_INPUTS` ở ô 1 nếu để chỗ khác).
2. Runtime → Change runtime type → **GPU** (T4 chạy được; L4/A100 của Colab Pro nhanh hơn nhiều).

**Cách chạy:** chạy lần lượt ô 1 → ô 5. Ô 4 chờ đến khi huấn luyện xong (nhiều giờ).

**Nếu bị ngắt kết nối hoặc hết phiên:** mở lại notebook, chạy lại ô 1, 2, 3 rồi ô 4. Tiến độ được đồng bộ lên
`MyDrive/SleepTCN_final_backup/` mỗi 5 phút: lần chạy đã xong được giữ, lần chạy dở tiếp tục từ checkpoint gần nhất.

Không sửa tệp nào trong repo. Nếu một ô dừng với `RuntimeError`, chụp vài dòng cuối gửi lại; **không xóa thư mục
backup trên Drive**.

In [ ]:
# Ô 1. Cấu hình và gắn Drive
import os, subprocess, time, json, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_INPUTS = Path("/content/drive/MyDrive/SleepTCN_upload")
BACKUP = Path("/content/drive/MyDrive/SleepTCN_final_backup")
REPO = Path("/content/SleepTCN")
PY = str(REPO / ".venv" / "bin" / "python")
ENV = {**os.environ, "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
CPUS = os.cpu_count() or 2
TRAIN_WORKERS = 3 if CPUS >= 6 else 2
LOADERS = 1 if CPUS >= 6 else 0
BUNDLES = {"a4_inputs_sleepedf.tar": "runs/a4/inputs_bundle_sleepedf.json",
           "final_core.tar": "runs/final/manifests/bundle_core.json",
           "final_holdout.tar": "runs/final/manifests/bundle_holdout.json"}
missing = [name for name in BUNDLES if not (DRIVE_INPUTS / name).is_file()]
if missing:
    raise RuntimeError(f"missing in {DRIVE_INPUTS}: {missing}")
BACKUP.mkdir(parents=True, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(f"CPUs {CPUS}: {TRAIN_WORKERS} training processes, {LOADERS} loader workers each")


def run(args, expect=None, cwd=None):
    print("$", " ".join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=cwd or REPO, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    output = "".join(lines)
    if process.wait() != 0:
        raise RuntimeError(f"command failed (exit {process.returncode})")
    if expect and expect not in output:
        raise RuntimeError(f"expected {expect!r} in the output")
    return output


def sh(command, cwd=None):
    return run(["bash", "-lc", command], cwd=cwd or "/content")

In [ ]:
# Ô 2. Mã nguồn, môi trường Python 3.11 + PyTorch 2.5.1/CUDA 12.1, dữ liệu, kiểm tra (khoảng 15-30 phút)
sh("command -v rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync)")
if not (REPO / ".git").is_dir():
    sh(f"git clone https://github.com/quanuiter/SleepTCN.git {REPO}")
run(["git", "status", "--short"])
if run(["git", "cat-file", "-t", "8af1d12"]).strip() != "commit":
    raise RuntimeError("repository history incomplete")

if not Path(PY).exists():
    sh("pip install --quiet 'uv==0.5.0'")
    sh(f"cd {REPO} && python -m uv venv --python-preference only-managed --python 3.11 --seed .venv")
    sh(f"cd {REPO} && {PY} -m pip install --quiet --upgrade pip && "
       f"{PY} -m pip install --quiet -r requirements/lock-cu121.txt && {PY} -m pip install --quiet -e . --no-deps")
run([PY, "-c", "import sys, torch; print(sys.version.split()[0], torch.__version__, torch.version.cuda, "
               "torch.cuda.is_available(), torch.cuda.get_device_name(0))"], expect="True")

for name, manifest in BUNDLES.items():
    if not (REPO / manifest).is_file():
        print("extracting", name, flush=True)
        sh(f"tar -xf '{DRIVE_INPUTS / name}' -C {REPO}")
run([PY, "scripts/run_amendment_a4.py", "verify-bundle"], expect="BUNDLE VERIFIED")
run([PY, "scripts/final_bundle.py", "verify", "--part", "core"], expect="BUNDLE VERIFIED")
run([PY, "scripts/final_bundle.py", "verify", "--part", "holdout"], expect="BUNDLE VERIFIED")

# restore progress from an earlier session
for sub in ("runs/a4", "runs/final"):
    if (BACKUP / sub).is_dir():
        sh(f"mkdir -p {REPO / sub} && rsync -a '{BACKUP / sub}/' '{REPO / sub}/'")
run(["git", "status", "--short"])
run([PY, "scripts/run_amendment_a4.py", "env-check", "--device", "cuda"], expect="ENVIRONMENT OK")
check = REPO / "runs/final/device_check.json"
if not (check.is_file() and json.loads(check.read_text())["passed"]):
    run([PY, "scripts/final_infer.py", "device-check", "--device", "cuda"], expect="DEVICE CHECK PASSED")
run([PY, "scripts/run_amendment_a4.py", "plan"])

In [ ]:
# Ô 3. Đồng bộ tiến độ lên Drive mỗi 5 phút (chạy nền; chạy lại ô này sau mỗi lần mở phiên mới)
SYNC = f'''while true; do
  for d in runs/a4 runs/final; do [ -d "{REPO}/$d" ] && mkdir -p "{BACKUP}/$d" && rsync -a "{REPO}/$d/" "{BACKUP}/$d/"; done
  date > "{BACKUP}/last_sync.txt"; sleep 300
done'''
if subprocess.run(["pgrep", "-f", "last_sync.txt"], capture_output=True).returncode != 0:
    subprocess.Popen(["bash", "-c", SYNC], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("sync started")
else:
    print("sync already running")


def sync_now():
    sh(f"for d in runs/a4 runs/final; do [ -d {REPO}/$d ] && mkdir -p {BACKUP}/$d && rsync -a {REPO}/$d/ {BACKUP}/$d/; done")

In [ ]:
# Ô 4. Kiểm tra tái lập, rồi huấn luyện 40 lần chạy (nền) và chờ đến khi xong. Chạy lại được sau khi ngắt.
def states():
    snippet = ("import json,sys; sys.path.insert(0,'src'); from pathlib import Path; "
               "from sleeptcn.amendment_a4 import load_a4_config,training_targets,training_state,repro_target; "
               "w=Path('.').resolve(); c,_=load_a4_config(w); n={}; "
               "[n.__setitem__(s,n.get(s,0)+1) for s in (training_state(w,t) for t in training_targets(c))]; "
               "print(json.dumps({'runs':n,'repro':training_state(w,repro_target())}))")
    result = subprocess.run([PY, "-c", snippet], cwd=REPO, env=ENV, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stdout + result.stderr)
    return json.loads(result.stdout.strip().splitlines()[-1])


def training_running():
    return subprocess.run(["pgrep", "-f", "run_amendment_a4.py train"], capture_output=True).returncode == 0


if states()["repro"] not in ("trained", "tested"):
    run([PY, "scripts/run_amendment_a4.py", "repro", "--device", "cuda"], expect="REPRODUCTION CHECK")
    sync_now()

s = states()
done = s["runs"].get("trained", 0) + s["runs"].get("tested", 0)
if done < 40 and not training_running():
    log = open(REPO / "runs/a4/train_driver.log", "a")
    subprocess.Popen([PY, "scripts/run_amendment_a4.py", "train", "--device", "cuda", "--workers", str(TRAIN_WORKERS),
                      "--num-workers", str(LOADERS)], cwd=REPO, env=ENV, stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True)
    print("training started")

while True:
    s = states()
    done = s["runs"].get("trained", 0) + s["runs"].get("tested", 0)
    print(time.strftime("%H:%M:%S"), s["runs"], f"({done}/40)", flush=True)
    if done == 40:
        break
    if not training_running():
        time.sleep(30)
        if not training_running() and states()["runs"].get("trained", 0) + states()["runs"].get("tested", 0) < 40:
            subprocess.run(["tail", "-n", "30", str(REPO / "runs/a4/train_driver.log")])
            raise RuntimeError("training stopped before all 40 runs finished; rerun this cell to resume")
    time.sleep(600)
sync_now()
print("ALL 40 A4 RUNS TRAINED")

In [ ]:
# Ô 5. Mở tập test một lần, suy luận Telemetry + SHHS1, xuất kết quả vào Drive
s = states()
if s["runs"].get("tested", 0) < 40:
    run([PY, "scripts/run_amendment_a4.py", "test"], expect="Preflight passed")
    run([PY, "scripts/run_amendment_a4.py", "test", "--device", "cuda", "--confirm", "OPEN-A4-LOCKED-TEST-ONCE"],
        expect="A4 TEST CAMPAIGN: complete")
    sync_now()
run([PY, "scripts/final_infer.py", "a4", "--device", "cuda"])
sync_now()
archive = Path("/content/final_results.tar.gz")
members = ["runs/a4/full", "runs/a4/monitoring", "runs/a4/test_campaign_a4.json", "runs/a4/reproduction_check.json",
           "runs/final"]
sh(f"cd {REPO} && tar -czf {archive} " + " ".join(m for m in members if (REPO / m).exists()))
shutil.copy(archive, BACKUP / archive.name)
print(sh(f"sha256sum {archive} {BACKUP / archive.name}"))
print("DONE: final_results.tar.gz is in", BACKUP, "- compare the two SHA-256 values, then send the file.")